# Exogenous Data Sources — Ingestion

Fetches macro (NDX, DXY), Fear & Greed, Binance funding rate, Google Trends,
and Reddit sentiment data, then merges everything onto the BTC date index.

**Run this notebook before re-running `02_feature_analysis.ipynb`.**

Missing-history policy: pre-inception gaps stay `NaN` with a `<feature>_missing`
flag (never imputed with a neutral value). Calendar/resolution gaps
(weekends for NDX/DXY, weekly resolution for Google Trends) are forward-filled,
each with their own `_missing` flag marking interpolated days.

In [1]:
import yfinance as yf

# Nasdaq Composite = ^IXIC (NOT ^NDX, which is the Nasdaq-100)
try:
    macro_tickers = ["^IXIC", "DX-Y.NYB"]
    macro_data = yf.download(macro_tickers, period="10y", interval="1d")
    macro_data.to_csv("../data/raw/yahoo_macro.csv")
    print(macro_data.shape)
    print(macro_data.tail())
except Exception as e:
    print(f"WARNING: yahoo_macro fetch failed ({e}) - downstream merge will treat this source as unavailable")

/var/folders/dy/8mkjsrmx33v0zw8njf1n2bxr0000gn/T/ipykernel_54880/487012001.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  macro_data = yf.download(macro_tickers, period="10y", interval="1d")


[                       0%                       ]

[*********************100%***********************]  2 of 2 completed

(2515, 10)
Price           Close                      High                      Low  \
Ticker       DX-Y.NYB         ^IXIC    DX-Y.NYB         ^IXIC   DX-Y.NYB   
Date                                                                       
2026-08-06  99.970001  26348.349609  100.019997  26499.419922  99.639999   
2026-08-07  99.599998  26690.619141  100.000000  26712.619141  99.400002   
2026-08-10  99.809998  26605.359375   99.830002  26724.630859  99.580002   
2026-08-11  99.820000  26445.449219   99.900002  26679.259766  99.730003   
2026-08-12  99.686996           NaN   99.902000           NaN  99.640999   

Price                          Open                 Volume                
Ticker             ^IXIC   DX-Y.NYB         ^IXIC DX-Y.NYB         ^IXIC  
Date                                                                      
2026-08-06  26208.429688  99.660004  26268.839844      0.0  8.936850e+09  
2026-08-07  26478.009766  99.940002  26534.660156      0.0  8.183970e+09  
2026-

In [2]:
import requests
import pandas as pd

try:
    resp = requests.get("https://api.alternative.me/fng/?limit=0&format=json", timeout=30)
    resp.raise_for_status()
    fng_raw = resp.json()["data"]

    fng = pd.DataFrame(fng_raw)
    fng["Date"] = pd.to_datetime(fng["timestamp"].astype(int), unit="s").dt.normalize()
    fng["fear_greed_value"] = fng["value"].astype(float)
    fng = fng.rename(columns={"value_classification": "fear_greed_classification"})
    fng = fng[["Date", "fear_greed_value", "fear_greed_classification"]].sort_values("Date")
    fng = fng.set_index("Date")
    fng.to_csv("../data/raw/fear_greed.csv")
    print(fng.shape)
    print(fng.head())
    print(fng.tail())
except Exception as e:
    print(f"WARNING: fear_greed fetch failed ({e}) - downstream merge will treat this source as unavailable")

(3111, 2)
            fear_greed_value fear_greed_classification
Date                                                  
2018-02-01              30.0                      Fear
2018-02-02              15.0              Extreme Fear
2018-02-03              40.0                      Fear
2018-02-04              24.0              Extreme Fear
2018-02-05              11.0              Extreme Fear
            fear_greed_value fear_greed_classification
Date                                                  
2026-08-08              30.0                      Fear
2026-08-09              31.0                      Fear
2026-08-10              30.0                      Fear
2026-08-11              29.0                      Fear
2026-08-12              27.0                      Fear
